# Notebook 16 - Final Report Generation

## What this notebook does
I programmatically fill the `paper_or_report/report.md` template with actual numeric results pulled from the saved metric tables. This creates a reproducible link between notebook outputs and manuscript text.

## Why this step matters
Copy-pasting numbers from notebooks into a Word document is error-prone. Generating the report programmatically ensures every number in the manuscript traces back to a saved CSV file.

## Input files expected
- `reports/tables/full_metrics_table.csv`
- `reports/tables/statistical_comparison.csv`
- `reports/tables/dp_sweep_results.csv`
- `reports/tables/site_statistics.csv`

## Output files created
- `paper_or_report/report.md`

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
from src.config import load_config
from src.paths import get_paths

cfg   = load_config()
paths = get_paths()
EPSILON = cfg["differential_privacy"]["target_epsilon"]
DELTA   = cfg["differential_privacy"]["target_delta"]
N_SITES = cfg["federated"]["num_clients"]
N_ROUNDS = cfg["federated"]["num_rounds"]
ALPHA   = cfg["federated"]["dirichlet_alpha"]
SEED    = cfg["project"]["random_seed"]
N_BOOT  = cfg["evaluation"]["n_bootstrap"]
C_NORM  = cfg["differential_privacy"]["max_grad_norm"]
DROPOUT = cfg["model"]["dropout"]
LOCAL_E = cfg["training"]["epochs_per_round"]
MARGIN  = 0.05

def load_table(path):
    p = Path(path)
    if p.exists():
        return pd.read_csv(p)
    print(f"  NOTE: {p.name} not found - placeholder values will appear.")
    return None

metrics_df = load_table(paths["tables"] / "full_metrics_table.csv")
stats_df   = load_table(paths["tables"] / "statistical_comparison.csv")
dp_df      = load_table(paths["tables"] / "dp_sweep_results.csv")
site_df    = load_table(paths["tables"] / "site_statistics.csv")

print("Tables loaded. Generating report...")

In [ ]:
def get_metric(df, model_name, metric_name):
    """
    Pull a formatted 'Estimate (95% CI: lower-upper)' string from the metrics table.
    Returns a placeholder string if the table or row is missing.
    """
    placeholder = "[PLACEHOLDER - run full pipeline]"
    if df is None:
        return placeholder
    mask = (
        df.get("Model", pd.Series(dtype=str)).str.strip() == model_name
    ) & (
        df["Metric"].str.contains(metric_name, case=False, na=False)
    )
    rows = df[mask]
    if len(rows) == 0:
        return placeholder
    r   = rows.iloc[0]
    est = r.get("Estimate", "?")
    ci  = r.get("95% CI", "?")
    return f"{est} (95% CI: {ci})"

def get_stat(df, test_name, col="statistic"):
    placeholder = "[PLACEHOLDER]"
    if df is None:
        return placeholder
    rows = df[df.get("test", pd.Series(dtype=str)).str.contains(test_name, case=False, na=False)]
    if len(rows) == 0:
        return placeholder
    return str(round(rows.iloc[0].get(col, "?"), 4))

# --- Centralised ---
c_auc   = get_metric(metrics_df, "Centralised",             "AUC-ROC")
c_prc   = get_metric(metrics_df, "Centralised",             "AUC-PRC")
c_sens  = get_metric(metrics_df, "Centralised",             "Sensitivity")
c_spec  = get_metric(metrics_df, "Centralised",             "Specificity")
c_ppv   = get_metric(metrics_df, "Centralised",             "PPV")
c_f1    = get_metric(metrics_df, "Centralised",             "F1")

# --- Federated No DP ---
fn_auc  = get_metric(metrics_df, "Federated (No DP)",       "AUC-ROC")
fn_sens = get_metric(metrics_df, "Federated (No DP)",       "Sensitivity")
fn_spec = get_metric(metrics_df, "Federated (No DP)",       "Specificity")

# --- Federated + DP ---
fd_auc  = get_metric(metrics_df, f"Federated + DP",         "AUC-ROC")
fd_sens = get_metric(metrics_df, f"Federated + DP",         "Sensitivity")
fd_spec = get_metric(metrics_df, f"Federated + DP",         "Specificity")
fd_f1   = get_metric(metrics_df, f"Federated + DP",         "F1")

# --- Statistical tests ---
mc_stat = get_stat(stats_df, "McNemar",        "statistic")
mc_p    = get_stat(stats_df, "McNemar",        "p_value")
auc_diff= get_stat(stats_df, "Bootstrap",      "statistic")
auc_p   = get_stat(stats_df, "Bootstrap",      "p_value")

print("Key results extracted for report:")
print(f"  Centralised AUC     : {c_auc}")
print(f"  FL No-DP AUC        : {fn_auc}")
print(f"  FL+DP AUC           : {fd_auc}")
print(f"  McNemar p           : {mc_p}")

In [ ]:
# Build the report text programmatically
report = f"""# FedTB-Nigeria: Federated Learning for TB Diagnosis from Chest X-Rays

**Authors**: [Insert author names]
**Institution**: African Institute for Mathematical Sciences (AIMS)
**Target Journal**: Nature Medicine
**Version**: 1.0 (draft - results auto-populated from pipeline outputs)

---

## Abstract

**Background**: Tuberculosis (TB) remains a leading cause of infectious disease mortality
in Nigeria. AI-assisted chest X-ray (CXR) screening can extend diagnostic capacity to
resource-limited settings, but centralising patient imaging data across teaching hospitals
raises serious privacy and logistical barriers.

**Methods**: We implemented FedTB-Nigeria, a federated learning (FL) system using
ResNet-18 trained via FedAvg across {N_SITES} simulated Nigerian teaching hospital sites
with the Flower framework. Client-level differential privacy was applied using Opacus
DP-SGD (target epsilon={EPSILON}, delta={DELTA:.0e}, gradient clip norm={C_NORM}).
Training data were partitioned using a Dirichlet distribution (alpha={ALPHA}) to
simulate realistic between-hospital heterogeneity. Models were evaluated on a
held-out global test set using AUC-ROC with 95% bootstrap confidence intervals
(n={N_BOOT} resamples) and compared against a centralised training baseline.

**Results**:
- Centralised baseline: AUC-ROC {c_auc}, Sensitivity {c_sens}, Specificity {c_spec}
- Federated (No DP):   AUC-ROC {fn_auc}, Sensitivity {fn_sens}, Specificity {fn_spec}
- Federated + DP (epsilon={EPSILON}): AUC-ROC {fd_auc}, Sensitivity {fd_sens}, Specificity {fd_spec}
- McNemar test (Centralised vs FL+DP): chi2={mc_stat}, p={mc_p}
- Bootstrap AUC difference: delta={auc_diff}, p={auc_p}

**Conclusions**: [To be completed after full experimental runs with real data.
Interpret conservatively and avoid overclaiming from simulated-site results.]

---

## 1. Introduction

### 1.1 Tuberculosis Burden in Nigeria

Nigeria carries one of the highest TB burdens globally, with an estimated incidence of
approximately 300 per 100,000 population (WHO Global TB Report, 2023). TB/HIV
co-infection is prevalent, complicating diagnosis and treatment. Chest X-ray screening
remains a frontline diagnostic tool in resource-limited settings where sputum culture
turnaround time is too slow for clinical decision-making.

### 1.2 AI-Assisted CXR Screening

Deep learning models trained on large CXR datasets have demonstrated performance
comparable to experienced radiologists for TB detection. However, the clinical
translation of such models is limited by:
1. The need for large, labelled datasets at each deployment site
2. Privacy regulations prohibiting centralisation of patient imaging data
3. Infrastructure differences across hospitals that cause distribution shift

### 1.3 Federated Learning and Differential Privacy

Federated Learning (FL; McMahan et al., 2017) enables collaborative model training
without sharing raw data. Each hospital trains locally on its own patients; only
gradient updates are shared with a central aggregator, which applies weighted averaging
(FedAvg) to produce an improved global model. Differential Privacy (Dwork et al.,
2006; Abadi et al., 2016) adds calibrated Gaussian noise to gradient updates, providing
the formal (epsilon, delta)-DP guarantee that bounds the information an adversary can
infer about any individual patient from the shared updates.

### 1.4 Research Gap

No prior published work combines FL + DP + ResNet-18 specifically for TB CXR
classification in the context of African teaching hospital simulation, with rigorous
statistical comparison against a centralised baseline and full error analysis.
FedTB-Nigeria addresses this gap.

---

## 2. Methods

### 2.1 Datasets

We combined two publicly available CXR datasets:

| Dataset | Source | N | TB+ | TB- |
|---|---|---|---|---|
| Montgomery County | US NLM | 138 | 58 | 80 |
| Shenzhen Hospital | Guanganmen Hospital / NLM | 662 | 336 | 326 |
| **Combined** | | **800** | **394** | **406** |

**Partitioning**: Stratified split into train (70%), validation (15%), test (15%).
Training data were further partitioned across {N_SITES} simulated hospital sites using
Dirichlet(alpha={ALPHA}). The test and validation sets were held globally (not partitioned).

**Dataset limitation**: These datasets originate from the USA and China respectively,
and do not represent Nigerian patient demographics, local TB strain diversity, or
TB/HIV co-infection patterns. This is a primary limitation of the present study
(see Section 5 and limitations.md).

### 2.2 Model Architecture

ResNet-18 pre-trained on ImageNet (He et al., 2016). The final classification head
was replaced with Dropout(p={DROPOUT}) -> Linear(512, 2). The backbone was frozen
for {cfg["model"]["freeze_backbone_epochs"]} epochs (feature extraction phase) then
unfrozen for full fine-tuning.

For DP training, BatchNorm layers were replaced with GroupNorm using Opacus's
ModuleValidator to satisfy per-sample gradient requirements.

### 2.3 Federated Learning

- Framework: Flower (flwr >= 1.6)
- Strategy: FedAvg (McMahan et al., 2017)
- Communication rounds: {N_ROUNDS}
- Local epochs per round: {LOCAL_E}
- Minimum participating clients per round: {cfg["federated"]["min_fit_clients"]}
- Site simulation: Dirichlet(alpha={ALPHA}) across {N_SITES} sites

### 2.4 Differential Privacy

- Library: Opacus >= 1.4
- Mechanism: DP-SGD with Gaussian noise
- Target epsilon: {EPSILON}
- Target delta: {DELTA:.0e}
- Max gradient norm (clipping): {C_NORM}
- Privacy accounting: Renyi Differential Privacy (RDP) accountant
- Budget allocation: equal split across {N_ROUNDS} rounds (sequential composition)

### 2.5 Evaluation

- Primary metric: AUC-ROC (threshold-independent)
- Secondary: AUC-PRC, Sensitivity, Specificity, PPV, NPV, F1, Balanced Accuracy
- Confidence intervals: 95% bootstrap CI (n={N_BOOT} resamples, percentile method)
- Decision threshold: Youden's J statistic (maximises Sensitivity + Specificity - 1)
- Statistical comparison: McNemar's test (paired, continuity-corrected),
  bootstrap AUC difference test
- Non-inferiority margin: delta_AUC < {MARGIN}

---

## 3. Results

### 3.1 Dataset and Site Characteristics

[See reports/tables/site_statistics.csv for per-site sample counts and TB+ rates.
Under Dirichlet(alpha={ALPHA}), sites exhibit moderate heterogeneity.]

### 3.2 Main Performance Results

See Table 1 (paper_or_report/tables/table1_metrics.tex) for the full metric table
with 95% bootstrap confidence intervals.

Summary:
- Centralised: AUC-ROC {c_auc} | F1 {c_f1}
- Federated (No DP): AUC-ROC {fn_auc}
- Federated + DP (epsilon={EPSILON}): AUC-ROC {fd_auc} | F1 {fd_f1}

### 3.3 Statistical Comparison

McNemar's test (Centralised vs FL+DP): chi2 = {mc_stat}, p = {mc_p}
Bootstrap AUC difference (Centralised - FL+DP): delta = {auc_diff}, p = {auc_p}

[Interpretation of non-inferiority to be completed after full pipeline run.]

### 3.4 Privacy-Utility Trade-off

[See Figure 2 and reports/tables/dp_sweep_results.csv.
Results show AUC as a function of epsilon across the sweep: epsilon in 1, 2, 4, 8.]

### 3.5 Error Analysis

[See Notebook 12 and reports/figures/error_analysis_examples.png.
False negatives (missed TB) are the highest clinical risk category.]

### 3.6 Robustness and Ablations

[See Notebook 13 and reports/tables/ablation_rounds.csv, ablation_alpha.csv.
Key finding: convergence achieved within N rounds (see ablation_rounds.csv).]

### 3.7 Fairness Analysis

[See Notebook 14 and reports/tables/fairness_analysis.csv.
Per-site AUC equity gap reported. Simulated site assignments noted as limitation.]

---

## 4. Discussion

[To be written after full experimental results are available.]

Points to address:
- Is FL+DP non-inferior to centralised training within the {MARGIN} AUC margin?
- What is the practical privacy-utility cost at epsilon={EPSILON}?
- Which hospital sites benefit most/least from federation?
- What types of TB cases does the model miss?
- How do results compare to prior FL-for-medical-imaging studies?
- What further validation is required before clinical deployment?

---

## 5. Limitations

1. **Non-Nigerian source data**: Montgomery and Shenzhen datasets do not represent
   Nigerian patients, TB strain diversity, or TB/HIV co-infection patterns. Performance
   on real Nigerian CXRs may differ substantially.

2. **Simulated hospital partitioning**: Dirichlet partitioning is a mathematical proxy.
   It does not capture real differences in imaging equipment, patient demographics,
   radiologist labelling practices, or institutional workflows.

3. **Absence of demographic metadata**: No age, sex, or HIV status information is
   available in these datasets for subgroup fairness analysis.

4. **Per-round DP composition**: We used sequential composition (epsilon_total <= N_rounds
   * epsilon_per_round). Tighter RDP composition across rounds would yield a smaller
   effective epsilon, potentially improving the privacy-utility trade-off.

5. **No external validation**: Results are evaluated on a hold-out split of the same
   source datasets. Independent validation on Nigerian hospital CXRs is required before
   any clinical translation.

6. **Reference standard uncertainty**: TB labels in source datasets are based on
   radiologist report and/or sputum culture, both of which have imperfect sensitivity
   and specificity, introducing label noise.

See `paper_or_report/limitations.md` for the extended limitations document.

---

## 6. Ethics and Bias

- No real patient data are used in this repository. All source datasets are publicly
  released for research use by the US National Library of Medicine.
- Known bias sources are documented in Notebook 14 and `limitations.md`.
- The system is designated a research prototype, not a clinical tool.
- Deployment would require ethics approval, regulatory clearance, and clinical validation
  at each Nigerian teaching hospital site.

---

## 7. Reproducibility

All experiments are fully reproducible:
- Random seed: {SEED} (fixed throughout)
- Environment: `conda env create -f environment.yml && conda activate fedtb_nigeria`
- Full pipeline: `python scripts/run_all.py`
- Individual notebooks: run in order 00-17

---

## 8. Data Availability

The Montgomery County and Shenzhen CXR datasets are freely available at:
  https://openi.nlm.nih.gov/imgs/collections/NLM-MontgomeryCXRSet.zip
  https://openi.nlm.nih.gov/imgs/collections/ChinaSet_AllFiles.zip

No patient data are stored in this repository.

---

## 9. Code Availability

Full code: https://github.com/YOUR_USERNAME/fedtb_nigeria (MIT License)

---

## References

[See paper_or_report/references.bib for full BibTeX bibliography.]

Key references:
- McMahan et al. (2017). Communication-Efficient Learning of Deep Networks from Decentralized Data. AISTATS.
- Abadi et al. (2016). Deep Learning with Differential Privacy. CCS.
- Dwork et al. (2006). Calibrating Noise to Sensitivity in Private Data Analysis. TCC.
- He et al. (2016). Deep Residual Learning for Image Recognition. CVPR.
- Beutel et al. (2020). Flower: A Friendly Federated Learning Research Framework. arXiv.
- Yousefpour et al. (2021). Opacus: User-Friendly Differential Privacy Library in PyTorch. arXiv.
- Lakhani & Sundaram (2017). Deep Learning at Chest Radiography. Radiology.
- Flores et al. (2021). Federated Learning for Medical Imaging. Radiology AI.
- WHO Global Tuberculosis Report (2023). Geneva: World Health Organization.
"""

report_path = paths["paper"] / "report.md"
with open(report_path, "w") as f:
    f.write(report)
print(f"Report written: {report_path}")
print(f"Character count: {len(report):,}")